## Homework \#1

In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

Q1. How many lesson pages are in the dataset?

In [3]:
len(files)

72

Q2. Indexing and searching

In [ ]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
index.fit(documents)

question = "How does the agentic loop keep calling the model until it stops?"
search_results = index.search(
    question,
    # boost_dict={"question": 2.0, "section": 0.5},
    # filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry 

In [7]:
search_results[0]['filename']

'01-agentic-rag/lessons/14-agentic-loop.md'

Q3. RAG

In [8]:
from dotenv import load_dotenv
load_dotenv()

from rag_helper import RAGBase
from openai import OpenAI

openai_client = OpenAI()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

answer, usage = assistant.rag(question)
print(answer)
print(usage)

It keeps a `while True` loop around the model call.

Each iteration:
1. Sends the current `messages` history to `openai_client.responses.create(...)`
2. Checks the response for any `function_call` items
3. Runs the tool and appends the tool output to `messages`
4. Sets `has_function_calls = True` if any tool was called

At the end of the loop, it breaks only when `has_function_calls == False`, meaning the model returned a message with no more tool calls.

So the loop keeps calling the model until the model stops asking for tools.
Response(id='resp_03774e00d68814ea006a399a6737b481a0bc988528f495e647', created_at=1782159975.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.4-mini-2026-03-17', object='response', output=[ResponseOutputMessage(id='msg_03774e00d68814ea006a399a67cfa481a0a1df0ec22e4dd1be', content=[ResponseOutputText(annotations=[], text='It keeps a `while True` loop around the model call.\n\nEach iteration:\n1. Sends the current `messages` hi

In [10]:
usage.usage.input_tokens, usage.usage.output_tokens, usage.usage.total_tokens

(7136, 127, 7263)

## Q4. Chunking

In [11]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
print(len(chunks))

295


## Q5. RAG w chunking

In [12]:
index_q5 = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
index_q5.fit(chunks)

assistant = RAGBase(
    index=index_q5,
    llm_client=openai_client,
)

answer, whole_response = assistant.rag(question)

print(whole_response.usage.input_tokens)

2319


(3x fewer)

## Q6. truning it into an agent

In [13]:
def search(query: str) -> list[dict]:
    # boost_dict = {'question': 3.0, 'section': 0.5}
    # filter_dict = {'course': 'llm-zoomcamp'}

    return index_q5.search(
        query,
        num_results=5,
        # boost_dict=boost_dict,
        # filter_dict=filter_dict
    )

# search_tool = {
#     "type": "function",
#     'name': 'search',
#     'description': 'Search the knowledge base of the course lessons for entries matching the given query.',
#     'parameters': {
#         "type": "object",
#         "properties": {
#             'query': {
#                 "type": "string",
#                 'description': 'Search query text to look up in the course knowledge base.'
#             }
#         },
#         "required": ["query"],
#         'additionalProperties': False
#     }
# }

question_q6 = "How does the agentic loop work, and how is it different from plain RAG?"

In [ ]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner

# Create tools
tools = Tools()

tools.add_tool(search)

# Create chat interface and client
chat_interface = IPythonChatInterface()

# Create and run chat assistant
runner = OpenAIResponsesRunner(
    tools=tools,
    developer_prompt="You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering.",
    chat_interface=chat_interface,
    llm_client = OpenAIClient(
            model='gpt-5.4-mini'
            client=openai_client
        )
)

runner.run()

AttributeError: 'OpenAI' object has no attribute 'send_request'